# FoF - Fear of Forgetting

My nostalgia has always led me to fear losing track and control over my memories, hence I've been trying to capture all my memories since high school through pictures and diary pages. I noticed that during uni my habits in this regard changed, with more things to do and less time to take pictures but i focused on writing more.

This study investigates the evolution of my **Fear of Forgetting habits**.

The first step consists in renaming the columns of my dataset, from natural language to snake case.

In [1]:
import pandas as pd

df = pd.read_csv('fof_dataset.csv')

rename_rules = {
    'university year': 'university_year',
    'iphone photos and videos': 'iphone_media_count',
    'camera photos and videos': 'camera_media_count',
    'journal pages': 'journal_pages',
    'my age': 'my_age',
    'academic period': 'academic_period',
    'exams prepared': 'exams_prepared',
    'hours of work': 'work_hours',
    'travel days': 'travel_days'
}

df = df.rename(columns=rename_rules)

df.head()

,month,iphone_media_count,camera_media_count,journal_pages,my_age,university_year,academic_period,exams_prepared,work_hours,travel_days
0,Sep-21,667,270,2,18,1,lectures,0,0,0
1,Oct-21,753,884,3,18,1,lectures,0,0,0
2,Nov-21,546,589,10,18,1,lectures,0,0,0
3,Dec-21,713,1181,5,19,1,lectures,1,0,0
4,Jan-22,558,507,2,19,1,exam session,1,0,0


## Did I actually stop taking as many pictures as the years went on?

In this next step I visualise if the number of pictures taken on my phone and camera decreased, or if it is only a feeling of mine.

In [66]:
import plotly.express as px
import pandas as pd

# make month string into date
df['date_parsed'] = pd.to_datetime(df['month'], format='%b-%y')
df = df.sort_values('date_parsed').reset_index(drop=True)

# create line graph
fig = px.line(
    df, 
    x='date_parsed', 
    y=['camera_media_count', 'iphone_media_count'],
    color_discrete_sequence=['#355c7d', '#c06c84'], 
    title='<b>The Digital Deceleration: Evolution of Photo Captures</b>',
    labels={'date_parsed': 'Timeline', 'value': 'Media Captured'}
)

legend_names = {
    'camera_media_count': 'Camera Photos & Videos', 
    'iphone_media_count': 'iPhone Photos & Videos'
}
fig.for_each_trace(lambda trace: trace.update(name = legend_names[trace.name]))

# make tooltip
for trace in fig.data:
    trace.hovertemplate = f"<b>{trace.name}</b>: %{{y:,}}<extra></extra>"
    trace.line.width = 3

# line graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_font_size=18,
    xaxis_title="University Years Timeline",
    yaxis_title="Monthly Media Captured",
    plot_bgcolor="white",
    width=1000,
    height=550,
    hovermode="x unified", 
    legend=dict(
        yanchor="top", 
        y=0.98, 
        xanchor="right", 
        x=0.98, 
        bgcolor="rgba(255,255,255,0.7)", 
        title_text=""
    )
)

fig.update_xaxes(showgrid=True, gridcolor='#EAEAEA', tickformat='%b-%y', dtick="M3", tickangle=45)
fig.update_yaxes(showgrid=True, gridcolor='#EAEAEA')

fig.show()

This graph shows how my idea was technically correct, as the number of total media captured across the two devices has indeed decreased with time. 
I notice that the camera count decreases more compared to the iphone one, because I carry my camera around mich less, as most days only consist in studying and not doing anything extra.  
It seems generally that the dicrease started in 2025. 

However, in the viz. there are peaks of media production in some specific months, and this is simply because I went on holiday, even for only a few days, hence photographing more anything new and yet unseen. 

## Did I stop taking as many pictures even when on vacation?

I know that on some months I went on vacation.
So first step is visualizing the vacation days compared for each month of my uni years.

In [51]:
import plotly.express as px
import pandas as pd

# create bar graph 
fig = px.bar(
    df,
    x='date_parsed',
    y='travel_days',
    color_discrete_sequence=['#c06c84'], 
    title="<b>Mapping the Breaks: Distribution of Travel Days Across 5 University Years</b><br><span style='font-size:14px; color:#555555;'><i>Monthly count of days spent traveling or on vacation (Sep 2021 – Apr 2026)</i></span>",
    labels={'date_parsed': 'Timeline', 'travel_days': 'Travel Days'}
)

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_font_size=22,
    plot_bgcolor="white",
    width=1100,
    height=500,
    hovermode="x" 
)

fig.update_xaxes(
    showgrid=False, 
    tickformat='%b-%y', 
    dtick="M3",
    tickangle=45,
    range=['2021-09-01', '2026-04-01'], 
    title_text="University Years Timeline"
)

fig.update_yaxes(
    showgrid=True, 
    gridcolor='#F3F3F3', 
    title_text="Number of Travel Days"
)

# tooltip
fig.update_traces(
    hovertemplate="<b>Date:</b> %{x|%b-%y}<br><b>Travel Days:</b> %{y} days<extra></extra>"
)

fig.show()

Now I want to understand if my habits in taking pictures to collect memories has changed during travels as well or not.
For doing so I want to calculate the number of photos taken for each travel day, by only considering the months in which I have actually travelled and calculating the total amount of media produced (phone+camerA) over the total days of travel of the month, so as to obtain a mean value to compare among the years.

In [60]:
import plotly.express as px
import pandas as pd

# only consider the months I travelled
travel_df = df[df['travel_days'] > 0].copy()

# photos x travel day index
travel_df['photos_per_travel_day'] = (travel_df['camera_media_count'] + travel_df['iphone_media_count']) / travel_df['travel_days']

# index yearly mean 
yearly_travel = travel_df.groupby('university_year')['photos_per_travel_day'].mean().reset_index()

# make year into string
yearly_travel['year_label'] = yearly_travel['university_year'].apply(lambda x: f"Year {x}")

# bar graph 
fig = px.bar(
    yearly_travel,
    x='year_label',
    y='photos_per_travel_day',
    text_auto='.0f', 
    color_discrete_sequence=['#355c7d'], 
    labels={'year_label': 'University Year', 'photos_per_travel_day': 'Avg Photos per Travel Day'}
)

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_text="<b>A Halved Photographic Footprint Over 5 Years of Travels</b><br><span style='font-size:14px; color:#555555;'><i>Average volume of photos and videos captured per individual travel day, aggregated by university year</i></span>",
    title_font_size=22,
    plot_bgcolor="white",
    width=1100,
    height=550,
    showlegend=False 
)

fig.update_xaxes(showgrid=False, title_text="University Years")
fig.update_yaxes(showgrid=True, gridcolor='#F0F0F0', title_text="Media Count per Travel Day")

fig.update_traces(
    textposition='outside', 
    textfont=dict(family="Times New Roman", size=14, color="black"),
    hovertemplate="<b>%{x}</b><br>Average: %{y:.1f} photos/day<extra></extra>"
)

fig.show()

The visualization suggests that my habit of media documenting my holidays also changed across the years. In the last year on vacation I took an average of half the pictures I used to take in my first years of uni.

My attachment to documenting anything may have decreased because I have grown up and tend to live things more fully and presently or may have changed due to some external factors that have influenced me. These factors will be analysed in the following section.

## What has changed in the last couple of years?

In this section I visualize the increase in busy-ness in the last two years, which I personally think is the cause for the dicrease of media captures, both on vacation and on the daily basis.
'+' note to academic periods

In [53]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd



# create 2 stacked graphs sharing x axis 
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=True, 
    vertical_spacing=0.08, 
    subplot_titles=("<b>Exams Prepared per Month</b>", "<b>Hours of Work per Month</b>")
)

# upper graph
fig.add_trace(
    go.Scatter(
        x=df['date_parsed'], y=df['exams_prepared'], 
        name="Exams Prepared", mode='lines+markers',
        line=dict(color='#355c7d', width=3.5),
        marker=dict(size=5),
        showlegend=True
    ),
    row=1, col=1
)

# bottom graph
fig.add_trace(
    go.Scatter(
        x=df['date_parsed'], y=df['work_hours'], 
        name="Hours of Work", mode='lines+markers',
        line=dict(color='#c06c84', width=3),
        marker=dict(size=5, symbol='diamond'),
        showlegend=True
    ),
    row=2, col=1
)

# data viz palette
period_colors = {
    'lectures': 'rgba(124, 143, 161, 0.18)',         
    'exam session': 'rgba(169, 135, 199, 0.18)',     
    'summer break': 'rgba(201, 143, 93, 0.18)',      
    'bachelor graduation': 'rgba(170, 92, 92, 0.18)'  
}

legend_marker_colors = {
    'lectures': '#7C8FA1',
    'exam session': '#A987C7',
    'summer break': '#C98F5D',
    'bachelor graduation': '#AA5C5C'
}

# legend and regions for academic periods
for period in period_colors.keys():
    fig.add_trace(
        go.Scatter(
            x=[None], y=[None], 
            mode='markers',
            marker=dict(color=legend_marker_colors[period], size=14, symbol='square'),
            name=f"Period: {period.title()}",
            showlegend=True
        )
    )

for i in range(len(df)):
    row_data = df.iloc[i]
    x0 = row_data['date_parsed']
    x1 = df.iloc[i+1]['date_parsed'] if i < len(df) - 1 else x0 + pd.DateOffset(months=1)
    
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor=period_colors[row_data['academic_period']],
        layer="below", line_width=0
    )

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_text="<b>The Compounding Burden: Five Years of Escalating Academic and Work Demands</b>",
    title_font_size=22,
    plot_bgcolor="white",
    width=1100, 
    height=650,
    hovermode="x unified",
    legend=dict(
        orientation="v",       
        yanchor="top",         
        y=1,
        xanchor="left",        
        x=1.03,                
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#E8E8E8",
        borderwidth=1
    )
)

fig.update_xaxes(showgrid=True, gridcolor='#F3F3F3', tickformat='%b-%y', dtick="M3", tickangle=45, range=['2021-08-01', '2026-05-01'], row=2, col=1, title_text="University Years Timeline")
fig.update_xaxes(showgrid=True, gridcolor='#F3F3F3', range=['2021-08-01', '2026-05-01'], row=1, col=1)
fig.update_yaxes(showgrid=True, gridcolor='#F3F3F3', title_text="Exam Count", row=1, col=1)
fig.update_yaxes(showgrid=True, gridcolor='#F3F3F3', title_text="Work Hours", row=2, col=1)

fig.show()



## Has my FoF evolved alongside the evolution of my responsibilities?

I know that my attachment to memories and fear of letting them go and not being able to recall them has not simply vanished but has been changing in the past years. 
I have taken up journaling, an habit I used to have as a child but gave up when I recieved my first phone and started taking pictures of everything. Since then, I have been taking up this habit periodically and then always ended up abandoning it.
It has become a strict part of my everyday life in the last months.

In this section I visualize how this habit of journaling has actually increased during these last years of university.

In [79]:
import plotly.graph_objects as go
import pandas as pd

# create line chart
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df['date_parsed'], 
        y=df['journal_pages'], 
        mode='lines+markers',
        line=dict(color='#c06c84', width=3.5), 
        marker=dict(size=5, symbol='circle')
    )
)

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_text="<b>A Steady Rise in Journaling Across Five University Years</b><br><span style='font-size:14px; color:#555555;'><i>Monthly volume of written journal pages (Sep 2021 – Apr 2026)</i></span>",
    title_font_size=22,
    plot_bgcolor="white",
    width=1100, 
    height=500,
    hovermode="x unified",
    showlegend=False  
)

fig.update_xaxes(
    showgrid=True, gridcolor='#F3F3F3', 
    tickformat='%b-%y', dtick="M3", tickangle=45, 
    range=['2021-09-01', '2026-04-01'], 
    title_text="University Years Timeline"
)

fig.update_yaxes(
    showgrid=True, gridcolor='#F3F3F3', 
    title_text="Number of Journal Pages"
)

fig.show()

Now I simply want to visualize the comparison between the evolutions of my 2 main methods of keeping track of my memories over time, as I know the media production has decreased but the journal pages increased. 

This difference is visualized across my five years at universtity, starting from when I was 18 to present day and comparing the average media produced and journal pages written for each year. 

The comparison here is based on the years of my growth, rather than the academic periods, hence it deals specifically withn personal growth rather than directly with my sources of responsability and mental overload. 

In [78]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# total number of media
df['total_media_production'] = df['iphone_media_count'] + df['camera_media_count']

# group by age
df['age_years'] = df['my_age'].astype(int)
age_summary = df.groupby('age_years')[['total_media_production', 'journal_pages']].mean().reset_index()

# make age into string
age_summary['age_label'] = age_summary['age_years'].apply(lambda x: f"Age {x}")

# diverging bar graph
fig = make_subplots(
    rows=1, cols=2, 
    shared_yaxes=True, 
    horizontal_spacing=0.08, 
)

# left bar graph 
fig.add_trace(
    go.Bar(
        x=age_summary['total_media_production'],
        y=age_summary['age_label'],
        orientation='h',
        name='Total Media Captured',
        marker=dict(color='#355c7d'), 
        text=age_summary['total_media_production'].round(0),
        textposition='inside',
        textfont=dict(family="Times New Roman", size=12, color="white"),
        hovertemplate="<b>%{y}</b><br>Avg Media: %{x:.0f} files/month<extra></extra>"
    ),
    row=1, col=1
)

# right bar graph
fig.add_trace(
    go.Bar(
        x=age_summary['journal_pages'],
        y=age_summary['age_label'],
        orientation='h',
        name='Journal Pages Written',
        marker=dict(color='#c06c84'), 
        text=age_summary['journal_pages'].round(1),
        textposition='inside',
        textfont=dict(family="Times New Roman", size=12, color="white"),
        hovertemplate="<b>%{y}</b><br>Avg Journal: %{x:.1f} pages/month<extra></extra>"
    ),
    row=1, col=2
)

fig.update_xaxes(autorange="reversed", row=1, col=1, title_text="← Monthly Average Media Production")
fig.update_xaxes(row=1, col=2, title_text="Monthly Average Journal Pages →")

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_text="<b>Media Inversion Across Age Milestones</b><br><span style='font-size:14px; color:#555555;'><i>As age and responsibilities advanced, digital media documentation contracted while personal journaling expanded</i></span>",
    title_font_size=22,
    plot_bgcolor="white",
    width=1100, 
    height=500,
    showlegend=False, 
    margin=dict(t=100, b=50, l=50, r=50)
)

fig.update_xaxes(showgrid=True, gridcolor='#F5F5F5')
fig.update_yaxes(
    showgrid=False, 
    tickfont=dict(family="Times New Roman", size=14, color="black"),
    row=1, col=1 
)

fig.show()

This shows that my FoF has not vanished with time and increasing responsabilities, it has simply changed shape and media, from digital to analog.

## Is there an actual correlation between the evolution of my Fof habits and the increase in responsabilities?

This section wills at verifying and visualizing the correlation between the increase in reposnabilities and the way my keeping track of memories has evolved.

The first step consists in discovering the correlation between the total media production and the hours of work and exams prepared.
First, I wanted to calculate my responsabilities by summing the hours of work and the load of the exams prepared. To do so, I reduced both to the same scale and summed them together, to have an overall **Academic & Professional Load index (APL)**. This index was compared with the total media produced (phone + camera) and the scatter plot was 
created.

In [88]:
import plotly.express as px
import pandas as pd

# min-max mormalization to create single scale for work and exams
min_work, max_work = df['work_hours'].min(), df['work_hours'].max()
min_exams, max_exams = df['exams_prepared'].min(), df['exams_prepared'].max()

df['work_norm'] = (df['work_hours'] - min_work) / (max_work - min_work) if max_work != min_work else 0
df['exams_norm'] = (df['exams_prepared'] - min_exams) / (max_exams - min_exams) if max_exams != min_exams else 0

# APL index
df['responsibility_index'] = ((df['work_norm'] + df['exams_norm']) / 2) * 100

# make uni years into strings
df['year_label'] = df['university_year'].apply(lambda x: f"Year {x}")

# scatter plot with regression line (OLS)
fig = px.scatter(
    df,
    x='responsibility_index',
    y='total_media_production',
    color='year_label',
    trendline="ols",                 
    trendline_scope="overall",       
    trendline_color_override="#222222", 
    labels={
        'responsibility_index': 'Total Responsibility Index (APL Scale 0-100)',
        'total_media_production': 'Total Media Count (Phone & Camera)',
        'year_label': 'University Year'
    },
    color_discrete_sequence=['#3861AE', '#5f7d95', '#a987c7', '#c06c84', '#8a334e'] 
)

# graph layout
fig.update_layout(
    font_family="Times New Roman",
    title_text="<b>The Crowding-Out Effect: Visual Memories Displaced by Academic & Professional Load</b><br><span style='font-size:14px; color:#555555;'><i>Scatter analysis of monthly media production against the combined index of work hours and exams prepared</i></span>",
    title_font_size=22,
    title_x=0.0, 
    plot_bgcolor="white",
    width=1100, 
    height=700,
    legend=dict(
        orientation="v",      
        yanchor="top",        
        y=1,
        xanchor="left",        
        x=1.03,                
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#E8E8E8",
        borderwidth=1
    )
)

fig.update_xaxes(showgrid=True, gridcolor='#F3F3F3', zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor='#F3F3F3', zeroline=False)

fig.update_traces(marker=dict(size=9, opacity=0.85, line=dict(width=0.5, color='white')))

# regression line styling
for trace in fig.data:
    if trace.mode == 'lines':
        trace.line.dash = 'dash'
        trace.line.width = 2.5
        trace.name = 'Overall Trendline'
        trace.showlegend = True

fig.show()

This trendline decreasing means negative correlation, hence with the increase of the responsabilities, the media production has decreased and the 2 factors show to be actually correlated.

"The Ordinary Least Squares (OLS) regression analysis reveals a clear negative correlation between academic/professional responsibilities and media documentation. With an R2 coefficient of 0.30, the Total Responsibility Index (APL) proves to be a impactful predictor, accounting on its own for nearly a third of the overall decline in photo and video production across the five-year timeline. The remaining 70% of variability can be attributed to the spontaneous and social nature of daily photography. This confirms that while institutional pressure acted as a powerful structural constraint, it did not entirely override my spontaneous habits." 

next step positive correlation APL-journal